# 8. QGIS-to-Python Utilities
# 8. QGISPython

Pure-Python replacements for common QGIS operations used in the
flood susceptibility workflow. These functions avoid the need for QGIS
or a GIS desktop environment, enabling fully reproducible analysis
from the command line.

## 1. Imports / 

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling
import jenkspy
import os

## 2. Euclidean Distance / 

Replaces QGIS Proximity (raster distance) tool. Computes Euclidean
distance from each pixel to the nearest pixel with value = 1 (source).

In [ ]:
def euclidean_distance_raster(source_raster_path, output_path,
 source_value=1):
 """Compute Euclidean distance from each pixel to nearest source pixel.

 Replaces QGIS: Processing > Raster Analysis > Proximity (Raster Distance).

 Parameters
 ----------
 source_raster_path : str
 Path to binary/source raster (e.g., river network).
 output_path : str
 Path for output distance raster (GeoTIFF, float32).
 source_value : int
 Pixel value(s) considered as source. Pixels with this value
 have distance = 0.

 Returns
 -------
 numpy.ndarray
 Distance array in raster units (meters for projected CRS).
 """
 with rasterio.open(source_raster_path) as src:
 source = src.read(1)
 profile = src.profile.copy()
 transform = src.transform
 cellsize_x = abs(transform.a) # pixel width
 cellsize_y = abs(transform.e) # pixel height

 # Identify source pixels
 source_mask = (source == source_value)

 if not source_mask.any():
 raise ValueError(f'No pixels with value {source_value} found')

 # Get coordinates of source pixels
 rows, cols = np.where(source_mask)
 # Convert to map coordinates
 xs, ys = rasterio.transform.xy(transform, rows, cols)
 source_coords = np.column_stack([xs, ys])

 print(f'Source pixels: {len(source_coords):,}')
 print(f'Raster shape: {source.shape}')

 # Create coordinate grids for all pixels
 yy, xx = np.meshgrid(
 np.arange(source.shape[0]) * cellsize_y,
 np.arange(source.shape[1]) * cellsize_x,
 indexing='ij'
 )
 coords_all = np.column_stack([xx.ravel(), yy.ravel()])

 # Compute distance to nearest source pixel
 # For large rasters, process in chunks to avoid memory issues
 chunk_size = 50000
 n_pixels = coords_all.shape[0]
 dist = np.full(n_pixels, np.inf, dtype=np.float32)

 for start in range(0, n_pixels, chunk_size):
 end = min(start + chunk_size, n_pixels)
 chunk = coords_all[start:end]
 # Distance from each chunk pixel to all source pixels
 # Shape: (chunk_size, n_source)
 d = np.sqrt(
 ((chunk[:, 0:1] - source_coords[:, 0:1].T) ** 2) +
 ((chunk[:, 1:2] - source_coords[:, 1:2].T) ** 2)
 )
 dist[start:end] = d.min(axis=1)
 print(f' Processed {min(end, n_pixels):,} / {n_pixels:,} pixels')

 dist = dist.reshape(source.shape)

 # Save output
 out_profile = profile.copy()
 out_profile.update(dtype=rasterio.float32, nodata=-1)
 with rasterio.open(output_path, 'w', **out_profile) as dst:
 dst.write(dist.astype(np.float32), 1)

 print(f'Saved: {output_path}')
 print(f' Distance range: {dist.min():.1f} - {dist.max():.1f} m')
 return dist


# Example usage:
# euclidean_distance_raster('data/raw/river_network.tif',
# 'data/raw/distance_to_river.tif',
# source_value=1)

## 3. NDVI Calculation / NDVI

Computes Normalized Difference Vegetation Index from Sentinel-2 bands.
Replaces QGIS: Raster Calculator with the NDVI formula.

$$\text{NDVI} = \frac{NIR - Red}{NIR + Red} = \frac{B8 - B4}{B8 + B4}$$

Sentinel-2 band references:
- B4: Red (665 nm), 10m resolution
- B8: NIR (842 nm), 10m resolution

In [ ]:
def calculate_ndvi(red_path, nir_path, output_path,
 nodata_value=-9999):
 """Calculate NDVI from Sentinel-2 Red (B4) and NIR (B8) bands.

 Replaces QGIS: Raster > Raster Calculator with (NIR - Red) / (NIR + Red).

 Parameters
 ----------
 red_path : str
 Path to Sentinel-2 B4 (Red, 665 nm) GeoTIFF.
 nir_path : str
 Path to Sentinel-2 B8 (NIR, 842 nm) GeoTIFF.
 output_path : str
 Path for output NDVI GeoTIFF (float32, range -1 to 1).
 nodata_value : int
 Value to assign for invalid pixels (division by zero).

 Returns
 -------
 numpy.ndarray
 NDVI array (float32).
 """
 with rasterio.open(red_path) as red_src:
 red = red_src.read(1).astype(np.float32)
 profile = red_src.profile.copy()
 red_nodata = red_src.nodata

 with rasterio.open(nir_path) as nir_src:
 nir = nir_src.read(1).astype(np.float32)
 nir_nodata = nir_src.nodata

 # Mask nodata
 if red_nodata is not None:
 red[red == red_nodata] = np.nan
 if nir_nodata is not None:
 nir[nir == nir_nodata] = np.nan

 # Compute NDVI
 denominator = nir + red
 ndvi = np.where(
 denominator != 0,
 (nir - red) / denominator,
 np.nan
 )

 # Clip to valid NDVI range
 ndvi = np.clip(ndvi, -1.0, 1.0)

 # Replace NaN with nodata for output
 ndvi_out = np.where(np.isfinite(ndvi), ndvi, nodata_value).astype(np.float32)

 # Save
 out_profile = profile.copy()
 out_profile.update(dtype=rasterio.float32, nodata=nodata_value)
 with rasterio.open(output_path, 'w', **out_profile) as dst:
 dst.write(ndvi_out, 1)

 valid = np.isfinite(ndvi)
 print(f'NDVI statistics (valid pixels: {valid.sum():,}):')
 print(f' Min: {np.nanmin(ndvi):.4f}')
 print(f' Max: {np.nanmax(ndvi):.4f}')
 print(f' Mean: {np.nanmean(ndvi):.4f}')
 print(f'Saved: {output_path}')
 return ndvi


# Example usage:
# calculate_ndvi('data/raw/B04.tif', 'data/raw/B08.tif', 'data/raw/ndvi.tif')

## 4. Raster Reprojection and Resampling / 

Replaces QGIS: Raster > Projections > Warp (Reproject).
Common use: reprojecting rasters to UTM Zone 47N (EPSG:32647)
at 8 m resolution.

In [ ]:
def reproject_raster(input_path, output_path,
 target_crs='EPSG:32647',
 target_resolution=None,
 resampling_method=Resampling.nearest):
 """Reproject and optionally resample a raster.

 Replaces QGIS: Processing > Raster Projections > Warp (Reproject).

 Parameters
 ----------
 input_path : str
 Path to input raster.
 output_path : str
 Path for output reprojected raster.
 target_crs : str
 Target coordinate reference system (default: EPSG:32647).
 target_resolution : tuple or None
 (pixel_width, pixel_height) in target CRS units.
 If None, uses calculated default.
 resampling_method : rasterio.warp.Resampling
 Resampling method (default: nearest for categorical,
 bilinear for continuous).

 Returns
 -------
 None
 """
 with rasterio.open(input_path) as src:
 src_crs = src.crs
 print(f'Input CRS: {src_crs}')
 print(f'Input shape: {src.width}x{src.height}')
 print(f'Input resolution: {src.res}')

 # Calculate default transform
 dst_transform, dst_width, dst_height = calculate_default_transform(
 src.crs, target_crs,
 src.width, src.height,
 *src.bounds,
 resolution=target_resolution
 )

 kwargs = src.meta.copy()
 kwargs.update({
 'crs': target_crs,
 'transform': dst_transform,
 'width': dst_width,
 'height': dst_height,
 })

 with rasterio.open(output_path, 'w', **kwargs) as dst:
 for i in range(1, src.count + 1):
 reproject(
 source=rasterio.band(src, i),
 destination=rasterio.band(dst, i),
 src_transform=src.transform,
 src_crs=src.crs,
 dst_transform=dst_transform,
 dst_crs=target_crs,
 resampling=resampling_method,
 )

 print(f'Output CRS: {target_crs}')
 print(f'Output shape: {dst_width}x{dst_height}')
 if target_resolution:
 print(f'Output resolution: {target_resolution}')
 print(f'Saved: {output_path}')


# Example usage:
# Reproject DEM to UTM 47N at 8m resolution
# reproject_raster(
# 'data/raw/elevation_wgs84.tif',
# 'data/raw/elevation.tif',
# target_crs='EPSG:32647',
# target_resolution=(8, 8)
# )
# For continuous rasters (DEM, NDVI, rainfall), use bilinear resampling:
# from rasterio.warp import Resampling
# reproject_raster(
# 'data/raw/ndvi_raw.tif',
# 'data/raw/ndvi.tif',
# target_crs='EPSG:32647',
# target_resolution=(8, 8),
# resampling_method=Resampling.bilinear
# )

## 5. Natural Breaks Classification / 

Jenks natural breaks on a raster, producing classified output.
Replaces QGIS: Raster > Raster Calculator + reclassification rules.

In [ ]:
def jenks_classify_raster(input_path, output_path,
 n_classes=5, nodata_value=0):
 """Classify a continuous raster using Jenks natural breaks.

 Replaces QGIS: Raster > Reclassify by Breaks Values.

 Parameters
 ----------
 input_path : str
 Path to input continuous raster (e.g., FSI).
 output_path : str
 Path for output classified raster (uint8, classes 1-N).
 n_classes : int
 Number of classes (default: 5).
 nodata_value : int
 Value for invalid pixels in output (default: 0).

 Returns
 -------
 tuple
 (breaks, labels) — list of break values and class labels.
 """
 with rasterio.open(input_path) as src:
 data = src.read(1).astype(np.float64)
 profile = src.profile.copy()
 nodata_in = src.nodata

 # Create validity mask
 valid = np.isfinite(data)
 if nodata_in is not None:
 valid &= (data != nodata_in)

 valid_data = data[valid]
 print(f'Valid pixels: {valid.sum():,} / {data.size:,}')

 # Compute Jenks natural breaks
 breaks = jenkspy.jenks_breaks(valid_data.tolist(), n_classes=n_classes)
 print(f'Break values: {[round(b, 4) for b in breaks]}')

 # Classify
 classified = np.full(data.shape, nodata_value, dtype=np.uint8)
 for i in range(len(breaks) - 1):
 lower = breaks[i]
 upper = breaks[i + 1]
 if i == len(breaks) - 2: # last class includes upper bound
 mask = valid & (data >= lower) & (data <= upper)
 else:
 mask = valid & (data >= lower) & (data < upper)
 classified[mask] = i + 1

 # Save
 out_profile = profile.copy()
 out_profile.update(dtype=rasterio.uint8, nodata=nodata_value)
 with rasterio.open(output_path, 'w', **out_profile) as dst:
 dst.write(classified, 1)

 # Print distribution
 print(f'\nClass distribution:')
 for i in range(1, n_classes + 1):
 count = (classified[valid] == i).sum()
 pct = count / valid.sum() * 100
 print(f' Class {i}: {count:>10,} pixels ({pct:>5.1f}%)')

 print(f'Saved: {output_path}')
 return breaks


# Example usage:
# jenks_classify_raster('outputs/fsi_continuous.tif',
# 'outputs/flood_susceptibility.tif',
# n_classes=5)

## 6. Raster Reclassification by Rules / 

Reclassify a raster using explicit value mapping or break rules.
Replaces QGIS: Raster > Reclassify.

In [ ]:
def reclassify_by_rules(input_path, output_path, rules,
 nodata_value=0, out_dtype=np.uint8):
 """Reclassify a raster using explicit mapping rules.

 Replaces QGIS: Processing > Raster Analysis > Reclassify by Layer.

 Parameters
 ----------
 input_path : str
 Path to input raster.
 output_path : str
 Path for output reclassified raster.
 rules : list of tuple
 Each rule: (old_min, old_max, new_value).
 Pixels with value in [old_min, old_max] get new_value.
 nodata_value : int
 Value for unclassified/invalid pixels.
 out_dtype : numpy dtype
 Output data type (default: uint8).

 Returns
 -------
 numpy.ndarray
 Reclassified array.
 """
 with rasterio.open(input_path) as src:
 data = src.read(1)
 profile = src.profile.copy()

 result = np.full(data.shape, nodata_value, dtype=out_dtype)

 for old_min, old_max, new_val in rules:
 mask = (data >= old_min) & (data <= old_max)
 result[mask] = new_val

 out_profile = profile.copy()
 out_profile.update(dtype=out_dtype, nodata=nodata_value)
 with rasterio.open(output_path, 'w', **out_profile) as dst:
 dst.write(result, 1)

 print(f'Reclassified {input_path} → {output_path}')
 unique, counts = np.unique(result[result != nodata_value], return_counts=True)
 for u, c in zip(unique, counts):
 print(f' Class {u}: {c:,} pixels')
 return result


# Example: reclassify elevation into 5 classes
# rules = [
# (0, 25, 5), # <25m → class 5 (most susceptible)
# (25, 50, 4),
# (50, 100, 3),
# (100, 200, 2),
# (200, 10000, 1), # >200m → class 1 (least susceptible)
# ]
# reclassify_by_rules('data/raw/elevation.tif',
# 'data/raw/elevation_reclass.tif',
# rules)

## 7. Summary / 

This notebook provides pure-Python replacements for QGIS operations:

| QGIS Tool | Python Function | Notebook |
|-----------|-----------------|----------|
| Proximity (Raster Distance) | `euclidean_distance_raster()` | 2 |
| Raster Calculator (NDVI) | `calculate_ndvi()` | 2 |
| Warp (Reproject) | `reproject_raster()` | 2 |
| Reclassify | `reclassify_by_rules()` | 4b |
| Jenks Classification | `jenks_classify_raster()` | 4b |

These functions use `rasterio` and `jenkspy` for I/O and classification,
eliminating the QGIS dependency while maintaining identical outputs.

**Required packages:** `rasterio`, `numpy`, `jenkspy` (see `requirements.txt`)